# GTM Intelligence Engine: Growth Signal Analysis

This notebook provides a local interface to explore the high-growth intent signals generated by the Bruin pipeline.

**Source Table**: `fct_growth_signals`  
**Database**: `./data/local_data.duckdb` (local file)

In [1]:
import duckdb
import pandas as pd

# Set display options for better analysis
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

## 1. Connect to DuckDB

In [8]:
# Connect to the persistent DuckDB file produced by the pipeline
con = duckdb.connect('../data/local_data.duckdb', read_only=True)

# List available tables to verify the run
con.execute("SHOW TABLES").df()

,name
0,fct_growth_signals


## 2. Query High-Growth Signals

Extract the aggregated signals for our tech keywords, sorted by `signal_count` to identify high-interest repositories.

In [9]:
query = """
SELECT 
    *
FROM fct_growth_signals
ORDER BY signal_count DESC
LIMIT 20;
"""

df_top_signals = con.execute(query).df()
df_top_signals

,signal_date,company_name,repo_name,repo_url,event_type,signal_count,intent_priority,processed_at
0,2026-03-23,affaan-m,affaan-m/everything-claude-code,https://api.github.com/repos/affaan-m/everything-claude-code,WatchEvent,440,High,2026-04-05 19:33:37.973940+02:00
1,2026-03-23,Crosstalk-Solutions,Crosstalk-Solutions/project-nomad,https://api.github.com/repos/Crosstalk-Solutions/project-nomad,WatchEvent,404,High,2026-04-05 19:33:37.973940+02:00
2,2026-03-24,affaan-m,affaan-m/everything-claude-code,https://api.github.com/repos/affaan-m/everything-claude-code,WatchEvent,396,High,2026-04-05 19:33:37.973940+02:00
3,2026-03-25,microsoft,microsoft/RustTraining,https://api.github.com/repos/microsoft/RustTraining,WatchEvent,377,High,2026-04-05 19:33:37.973940+02:00
4,2026-03-25,affaan-m,affaan-m/everything-claude-code,https://api.github.com/repos/affaan-m/everything-claude-code,WatchEvent,360,High,2026-04-05 19:33:37.973940+02:00
5,2026-03-26,affaan-m,affaan-m/everything-claude-code,https://api.github.com/repos/affaan-m/everything-claude-code,WatchEvent,265,High,2026-04-05 19:33:37.973940+02:00
6,2026-03-24,microsoft,microsoft/RustTraining,https://api.github.com/repos/microsoft/RustTraining,WatchEvent,255,High,2026-04-05 19:33:37.973940+02:00
7,2026-03-24,Crosstalk-Solutions,Crosstalk-Solutions/project-nomad,https://api.github.com/repos/Crosstalk-Solutions/project-nomad,WatchEvent,240,High,2026-04-05 19:33:37.973940+02:00
8,2026-03-23,pascalorg,pascalorg/editor,https://api.github.com/repos/pascalorg/editor,WatchEvent,230,High,2026-04-05 19:33:37.973940+02:00
9,2026-03-25,pascalorg,pascalorg/editor,https://api.github.com/repos/pascalorg/editor,WatchEvent,228,High,2026-04-05 19:33:37.973940+02:00


In [10]:
# DK - row count for debug:

count_rows = """
SELECT COUNT(*)
FROM fct_growth_signals;
"""

df_rows = con.execute(count_rows).df()
df_rows # 7851 if we collected 1 day worth of data for 2026-03-19 as
# docker compose run --rm bruin bruin run . 
# this should be our default date to start with

,count_star()
0,28874


In [14]:
# DK - list individual dates we collected so far:

collected_dates = """
SELECT DISTINCT signal_date
FROM fct_growth_signals
ORDER BY 1;
"""

df_dates = con.execute(collected_dates).df()
df_dates

,signal_date
0,2026-03-23
1,2026-03-24
2,2026-03-25
3,2026-03-26


## 3. Analysis by Intent Priority

In [15]:
df_priority = con.execute("""
    SELECT 
        intent_priority, 
        COUNT(*) as signal_count 
    FROM fct_growth_signals 
    GROUP BY 1
""").df()

df_priority

,intent_priority,signal_count
0,Medium,541
1,Low,28305
2,High,28


## 4. Resource Cleanup

In [16]:
con.close() # close duckdb connection explicitly - just in  case...